# 04 — 模型總比較表

整合 RF、LR、TabPFN 三個模型的評估結果，輸出統一比較表並選出最佳模型。

> 請先依序執行 01、02、03 Notebook，確認以下 CSV 均已產生：
> - `rf_metrics.csv`
> - `lr_metrics.csv`
> - `tabpfn_metrics.csv`

In [7]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = next(
    path for path in [Path('..').resolve(), Path('.').resolve()]
    if (path / 'outputs' / 'metrics').exists()
)
METRICS_DIR = PROJECT_ROOT / 'outputs' / 'metrics'

## 1. 載入各模型指標

In [8]:
files = {
    'rf_metrics.csv':     'rf_metrics.csv',
    'lr_metrics.csv':     'lr_metrics.csv',
    'tabpfn_metrics.csv': 'tabpfn_metrics.csv',
}

frames = []
for label, fname in files.items():
    p = METRICS_DIR / fname
    if p.exists():
        frames.append(pd.read_csv(p))
    else:
        print(f'  [MISSING] {fname} — 請先執行對應 Notebook')

if not frames:
    raise FileNotFoundError('沒有找到任何 metrics CSV，請先執行 01–03 Notebook。')

summary = pd.concat(frames, ignore_index=True)
print(f'共載入 {len(summary)} 筆模型紀錄')

共載入 4 筆模型紀錄


## 2. 總比較表（依 AUC 由高到低）

In [9]:
summary_sorted = summary.sort_values('AUC', ascending=False).reset_index(drop=True)
summary_sorted.index += 1  # 排名從 1 開始
summary_sorted

,Model,Feature_Set,Accuracy,AUC,F1,Brier_Score
1,Random Forest,All 33,0.6927,0.7679,0.7277,0.2029
2,TabPFN,All 33,0.6927,0.7494,0.7208,0.2059
3,TabPFN,Diff 11,0.6872,0.7415,0.7186,0.2085
4,Logistic Regression,Diff 11,0.5335,0.5380,0.6640,0.2472


## 3. 最佳模型

In [10]:
best = summary_sorted.iloc[0]
print('=== 目前最佳模型（依 AUC）===')
print(f'  模型       : {best["Model"]}')
print(f'  特徵集     : {best["Feature_Set"]}')
print(f'  Accuracy   : {best["Accuracy"]}')
print(f'  AUC        : {best["AUC"]}')
print(f'  F1         : {best["F1"]}')
print(f'  Brier Score: {best["Brier_Score"]} （越低越好）')

=== 目前最佳模型（依 AUC）===
  模型       : Random Forest
  特徵集     : All 33
  Accuracy   : 0.6927
  AUC        : 0.7679
  F1         : 0.7277
  Brier Score: 0.2029 （越低越好）


## 4. 各指標排名

In [11]:
for metric in ['Accuracy', 'AUC', 'F1']:
    asc = False
    ranked = summary.sort_values(metric, ascending=asc)
    print(f'\n--- {metric} 排名 ---')
    for i, row in enumerate(ranked.itertuples(), 1):
        print(f'  {i}. {row.Model} ({row.Feature_Set}): {getattr(row, metric)}')

# Brier Score：越低越好
ranked_b = summary.sort_values('Brier_Score', ascending=True)
print('\n--- Brier Score 排名（越低越好）---')
for i, row in enumerate(ranked_b.itertuples(), 1):
    print(f'  {i}. {row.Model} ({row.Feature_Set}): {row.Brier_Score}')


--- Accuracy 排名 ---
  1. Random Forest (All 33): 0.6927
  2. TabPFN (All 33): 0.6927
  3. TabPFN (Diff 11): 0.6872
  4. Logistic Regression (Diff 11): 0.5335

--- AUC 排名 ---
  1. Random Forest (All 33): 0.7679
  2. TabPFN (All 33): 0.7494
  3. TabPFN (Diff 11): 0.7415
  4. Logistic Regression (Diff 11): 0.538

--- F1 排名 ---
  1. Random Forest (All 33): 0.7277
  2. TabPFN (All 33): 0.7208
  3. TabPFN (Diff 11): 0.7186
  4. Logistic Regression (Diff 11): 0.664

--- Brier Score 排名（越低越好）---
  1. Random Forest (All 33): 0.2029
  2. TabPFN (All 33): 0.2059
  3. TabPFN (Diff 11): 0.2085
  4. Logistic Regression (Diff 11): 0.2472


## 5. 輸出總比較表

In [12]:
out_path = METRICS_DIR / 'model_comparison.csv'
summary_sorted.to_csv(out_path, index=True, index_label='Rank')
print(f'model_comparison.csv 已輸出（{len(summary_sorted)} 個模型）')

model_comparison.csv 已輸出（4 個模型）
